 # Lab:  Transfer Learning with a Pre-Trained Deep Neural Network

As we discussed earlier, state-of-the-art neural networks involve millions of parameters that are prohibitively difficult to train from scratch.  In this lab, we will illustrate a powerful technique called *fine-tuning* where we start with a large pre-trained network and then re-train only the final layers to adapt to a new task.  The method is also called *transfer learning* and can produce excellent results on very small datasets with very little computational time.  

This lab is based partially on this
[excellent blog](https://blog.keras.io/building-powerful-image-classification-models-using-very-little-data.html).  In performing the lab, you will learn to:
* Build a custom image dataset
* Fine tune the final layers of an existing deep neural network for a new classification task.
* Load images with a `DataGenerator`.

The lab has two versions:
* *CPU version*:  In this version, you use lower resolution images so that the lab can be performed on your laptop.  The resulting accuracy is lower.  The code will also take considerable time to execute.
* *GPU version*:  This version uses higher resolution images but requires a GPU instance. See the [notes](../GCP/getting_started.md) on setting up a GPU instance on Google Cloud Platform.  The GPU training is much faster (< 1 minute).  

**MS students must complete the GPU version** of this lab.

## Create a Dataset

In this example, we will try to develop a classifier that can discriminate between two classes:  `cars` and `bicycles`.  One could imagine this type of classifier would be useful in vehicle vision systems.   The first task is to build a dataset.  

TODO:  Create training and test datasets with:
* 1000 training images of cars
* 1000 training images of bicylces
* 300 test images of cars
* 300 test images of bicylces
* The images don't need to be the same size.  But, you can reduce the resolution if you need to save disk space.

The images should be organized in the following directory structure:

    ./train
        /car
           car_0000.jpg
           car_0001.jpg
           ...
           car_0999.jpg
        /bicycle
           bicycle_0000.jpg
           bicycle_0001.jpg
           ...
           bicycle_0999.jpg
    ./test
        /car
           car_1001.jpg
           car_1001.jpg
           ...
           car_1299.jpg
        /bicycle
           bicycle_1000.jpg
           bicycle_1001.jpg
           ...
           bicycle_1299.jpg
           
The naming of the files within the directories does not matter.  The `ImageDataGenerator` class below will find the filenames.  Just make sure there are the correct number of files in each directory.
           
A nice automated way of building such a dataset if through the [FlickrAPI](demo2_flickr_images.ipynb).  Remember that if you run the FlickrAPI twice, it may collect the same images.  So, you need to run it once and split the images into training and test directories.         
        

In [ ]:
# FREE ALTERNATIVES TO FLICKR API:
# Option 1: Unsplash API (FREE - register at https://unsplash.com/developers)
# Option 2: Manual download - see instructions below
# Option 3: Use a pre-existing dataset

import os
import skimage.io
import skimage.transform
import requests
from io import BytesIO
import warnings
import random
import time

# Create directory structure
os.makedirs('./train/car', exist_ok=True)
os.makedirs('./train/bicycle', exist_ok=True)
os.makedirs('./test/car', exist_ok=True)
os.makedirs('./test/bicycle', exist_ok=True)

def download_images_unsplash(keyword, n_total, train_dir, test_dir, n_train=1000, n_test=300, access_key=None):
    """
    Download images from Unsplash API (FREE - register at https://unsplash.com/developers)
    """
    if not access_key:
        print(f"\n⚠️  No API key provided for '{keyword}'.")
        print("To download images automatically:")
        print("1. Go to https://unsplash.com/developers (FREE registration)")
        print("2. Create a new application")
        print("3. Copy your 'Access Key'")
        print("4. Set: access_key = 'YOUR_ACCESS_KEY_HERE'")
        print("\nAlternatively, manually download images and place them in:")
        print(f"  - {train_dir}/ (need {n_train} images)")
        print(f"  - {test_dir}/ (need {n_test} images)")
        return
    
    print(f"Downloading {n_total} images for '{keyword}' using Unsplash API...")
    
    images_downloaded = []
    page = 1
    per_page = 30  # Unsplash allows up to 30 per page
    
    base_url = "https://api.unsplash.com/search/photos"
    headers = {"Authorization": f"Client-ID {access_key}"}
    
    while len(images_downloaded) < n_total:
        try:
            params = {"query": keyword, "per_page": per_page, "page": page}
            response = requests.get(base_url, headers=headers, params=params, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                results = data.get('results', [])
                if not results:
                    print(f"No more results for '{keyword}'")
                    break
                    
                for photo in results:
                    url = photo.get('urls', {}).get('regular') or photo.get('urls', {}).get('small')
                    if url:
                        try:
                            img_response = requests.get(url, timeout=10)
                            if img_response.status_code == 200:
                                file = BytesIO(img_response.content)
                                im = skimage.io.imread(file)
                                if len(im.shape) == 3:  # Color image
                                    images_downloaded.append(im)
                                    if len(images_downloaded) % 50 == 0:
                                        print(f"  Downloaded {len(images_downloaded)}/{n_total} images...")
                                    if len(images_downloaded) >= n_total:
                                        break
                        except Exception as e:
                            continue
                            
                page += 1
                time.sleep(0.5)  # Rate limiting (50 requests per hour for free tier)
            else:
                print(f"API Error: {response.status_code}")
                break
                
        except Exception as e:
            print(f"Error: {e}")
            break
    
    if len(images_downloaded) == 0:
        print(f"⚠️  No images downloaded for '{keyword}'.")
        return
    
    print(f"Successfully downloaded {len(images_downloaded)} images")
    
    # Shuffle and split into train/test
    random.shuffle(images_downloaded)
    
    # Save training images
    for idx, im in enumerate(images_downloaded[:n_train]):
        filename = f'{train_dir}/{keyword}_{idx:04d}.jpg'
        # Resize if needed
        if im.shape[0] > 300 or im.shape[1] > 300:
            im = skimage.transform.resize(im, (224, 224), mode='constant', anti_aliasing=True)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                im = skimage.img_as_ubyte(im)
        skimage.io.imsave(filename, im)
    
    # Save test images
    for idx, im in enumerate(images_downloaded[n_train:n_train+n_test]):
        filename = f'{test_dir}/{keyword}_{idx:04d}.jpg'
        # Resize if needed
        if im.shape[0] > 300 or im.shape[1] > 300:
            im = skimage.transform.resize(im, (224, 224), mode='constant', anti_aliasing=True)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                im = skimage.img_as_ubyte(im)
        skimage.io.imsave(filename, im)
    
    print(f"Saved {min(n_train, len(images_downloaded))} training images to {train_dir}")
    print(f"Saved {min(n_test, max(0, len(images_downloaded)-n_train))} test images to {test_dir}")

# ============================================================================
# GET YOUR FREE UNSPLASH API KEY:
# 1. Go to: https://unsplash.com/developers
# 2. Click "Register as a developer" (FREE)
# 3. Create a new application
# 4. Copy your "Access Key"
# 5. Paste it below:
# ============================================================================
access_key = None  # Replace with: "YOUR_ACCESS_KEY_HERE"

# Download images for both classes
# Note: For testing, you can reduce numbers (e.g., 50 train, 20 test)
download_images_unsplash('car', 1300, './train/car', './test/car', 
                         n_train=1000, n_test=300, access_key=access_key)
download_images_unsplash('bicycle', 1300, './train/bicycle', './test/bicycle', 
                         n_train=1000, n_test=300, access_key=access_key)

print("\n" + "="*60)
print("Dataset creation complete!")
print("="*60)


## Loading a Pre-Trained Deep Network

We follow the [VGG16 demo](./demo3_vgg16.ipynb) to load a pre-trained deep VGG16 network.  First, run a command to verify your instance is connected to a GPU.

In [3]:
# TODO verify instance is connected to a GPU
import tensorflow as tf
tf.config.list_physical_devices('GPU') #empty because using CPU


[]

Now load the appropriate tensorflow packages.

In [4]:
from tensorflow.keras import applications
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import optimizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dropout, Flatten, Dense


We also load some standard packages.

In [5]:
import numpy as np
import matplotlib.pyplot as plt

Clear the Keras session.

In [6]:
# TODO
tf.keras.backend.clear_session()

Set the dimensions of the input image.  The sizes below would work on a GPU machine.  But, if you have a CPU image, you can use a smaller image size, like `64 x 64`.

In [7]:
# TODO:  Set to smaller values if you are using a CPU.  
# Otherwise, do not change this code.
# For CPU version, use 64x64. For GPU version, use 150x150
nrow = 64
ncol = 64

Now we follow the [VGG16 demo](./vgg16.ipynb) and load the deep VGG16 network.  Alternatively, you can use any other pre-trained model in keras.  When using the `applications.VGG16` method you will need to:
* Set `include_top=False` to not include the top layer
* Set the `image_shape` based on the above dimensions.  Remember, `image_shape` should be `height x width x 3` since the images are color.

In [8]:
# TODO:  Load the VGG16 network
input_shape = (nrow, ncol, 3)
base_model = applications.VGG16(weights='imagenet', include_top=False, input_shape=input_shape)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


To create now new model, we create a Sequential model.  Then, loop over the layers in `base_model.layers` and add each layer to the new model.

In [9]:
# Create a new model
model = Sequential()

# TODO:  Loop over base_model.layers and add each layer to model
for layer in base_model.layers:
    model.add(layer)


Next, loop through the layers in `model`, and freeze each layer by setting `layer.trainable = False`.  This way, you will not have to *re-train* any of the existing layers.

In [10]:
# TODO
for layer in model.layers:
    layer.trainable = False


Now, add the following layers to `model`:
* A `Flatten()` layer which reshapes the outputs to a single channel.
* A fully-connected layer with 256 output units and `relu` activation
* A `Dropout(0.5)` layer.
* A final fully-connected layer.  Since this is a binary classification, there should be one output and `sigmoid` activation.

In [11]:
# TODO
model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))


Print the model summary.  This will display the number of trainable parameters vs. the non-trainable parameters.

In [12]:
# TODO
print(model.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ block1_conv1 (Conv2D)           │ (None, 64, 64, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 16, 16, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 16, 16, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 8, 8, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 4, 4, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 4, 4, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 4, 4, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 2, 2, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,239,489 (58.13 MB)

 Trainable params: 524,801 (2.00 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

None


## Using Generators to Load Data

Up to now, the training data has been represented in a large matrix.  This is not possible for image data when the datasets are very large.  For these applications, the `keras` package provides a `ImageDataGenerator` class that can fetch images on the fly from a directory of images.  Using multi-threading, training can be performed on one mini-batch while the image reader can read files for the next mini-batch. The code below creates an `ImageDataGenerator` for the training data.  In addition to the reading the files, the `ImageDataGenerator` creates random deformations of the image to expand the total dataset size.  When the training data is limited, using data augmentation is very important.

In [13]:
train_data_dir = './train'
batch_size = 32
train_datagen = ImageDataGenerator(rescale=1./255,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)
train_generator = train_datagen.flow_from_directory(
                        train_data_dir,
                        target_size=(nrow,ncol),
                        batch_size=batch_size,
                        class_mode='binary')

FileNotFoundError: [Errno 2] No such file or directory: './train'

Now, create a similar `test_generator` for the test data.

In [13]:
# TODO
# test_generator = ...

The following function displays images that will be useful below.

In [14]:
# Display the image
def disp_image(im):
    if (len(im.shape) == 2):
        # Gray scale image
        plt.imshow(im, cmap='gray')    
    else:
        # Color image.  
        im1 = (im-np.min(im))/(np.max(im)-np.min(im))*255
        im1 = im1.astype(np.uint8)
        plt.imshow(im1)    
        
    # Remove axis ticks
    plt.xticks([])
    plt.yticks([])

To see how the `train_generator` works, use the `train_generator.next()` method to get a minibatch of data `X,y`.  Display the first 8 images in this mini-batch and label the image with the class label.  You should see that bicycles have `y=0` and cars have `y=1`.

In [15]:
# TODO

## Train the Model

Compile the model.  Select the correct `loss` function, `optimizer` and `metrics`.  Remember that we are performing binary classification.

In [16]:
# TODO.
# model.compile(...)

When using an `ImageDataGenerator`, we have to set two parameters manually:
* `steps_per_epoch =  training data size // batch_size`
* `validation_steps =  test data size // batch_size`

We can obtain the training and test data size from `train_generator.n` and `test_generator.n`, respectively.

In [17]:
# TODO

Now, we run the fit.  If you are using a CPU on a regular laptop, each epoch will take about 3-4 minutes, so you should be able to finish 5 epochs or so within 20 minutes.  On a reasonable GPU, even with the larger images, it will take about 10 seconds per epoch.
* If you use `(nrow,ncol) = (64,64)` images, you should get around 90% accuracy after 5 epochs.
* If you use `(nrow,ncol) = (150,150)` images, you should get around 96% accuracy after 5 epochs.  But, this will need a GPU.

You will get full credit for either version.  With more epochs, you may get slightly higher, but you will have to play with the damping.

Remember to record the history of the fit, so that you can plot the training and validation accuracy curve.

In [ ]:
nepochs = 5  # Number of epochs

# Call the fit_generator function
hist = model.fit_generator(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=nepochs,
    validation_data=test_generator,
    validation_steps=validation_steps)

In [ ]:
# Plot the training accuracy and validation accuracy curves on the same figure.

# TO DO

## Plotting the Error Images

Now try to plot some images that were in error:

*  Generate a mini-batch `Xts,yts` from the `test_generator.next()` method
*  Get the class probabilities using the `model.predict( )` method and compute predicted labels `yhat`.
*  Get the images where `yts[i] ~= yhat[i]`.
*  If you did not get any prediction error in one minibatch, run it multiple times.
*  After you a get a few error images (say 4-8), plot the error images with the true labels and class probabilities predicted by the classifie

In [ ]:
# TO DO